# GPT-2 and GPT-3 Basic Architecture in PyTorch

This notebook implements the same educational task as the NumPy version, but uses PyTorch modules.

The implementation covers:

- Reading `/content/textdata.txt`
- Building a simple character-level tokenizer
- Encoding text into token IDs
- Implementing a GPT-style decoder-only Transformer
- Creating toy GPT-2-like and GPT-3-like models
- Predicting the next token
- Generating a chosen length of text
- Optionally running a small training loop

Real GPT-2 and GPT-3 use the same broad decoder-only Transformer idea, but at much larger scale and with subword tokenization rather than character tokenization.

## 1. Imports and Reproducibility

In [1]:
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

# Fixed seed makes random initialization and sampling easier to reproduce.
torch.manual_seed(7)

# Use GPU if available; otherwise use CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## 2. Load the Text Dataset

The requested path is a common Google Colab path. Upload the file to this location before running the cell.

In [4]:
TEXT_PATH = Path("/content/textdata.txt")

def load_text_dataset(path=TEXT_PATH):
    """Read a UTF-8 text file from disk."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. In Colab, upload textdata.txt to this path first."
        )
    return path.read_text(encoding="utf-8", errors="replace")


text_data = load_text_dataset(TEXT_PATH)
print(f"Loaded {len(text_data):,} characters")
print(text_data[:500])

Loaded 161 characters
Welcome to Natural Language Processing  
It is one of the most exciting research areas as of today  
We will see how Python can be used to work with text files. 


## 3. Character-Level Tokenizer

GPT-2 and GPT-3 use subword tokenizers. For a teaching notebook, character-level tokenization is easier to inspect: every unique character becomes one token.

In [5]:
class CharTokenizer:
    """Minimal character tokenizer for educational language modeling."""

    def __init__(self, text):
        self.chars = sorted(set(text))
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

    @property
    def vocab_size(self):
        return len(self.chars)

    def encode(self, text):
        """Convert text into a list of integer token IDs."""
        return [self.stoi[ch] for ch in text if ch in self.stoi]

    def decode(self, ids):
        """Convert token IDs back into text."""
        return "".join(self.itos[int(i)] for i in ids)


tokenizer = CharTokenizer(text_data)
data_ids = torch.tensor(tokenizer.encode(text_data), dtype=torch.long)

print("Vocabulary size:", tokenizer.vocab_size)
print("Encoded dataset shape:", tuple(data_ids.shape))
print("First 100 decoded characters:")
print(tokenizer.decode(data_ids[:100].tolist()))

Vocabulary size: 29
Encoded dataset shape: (161,)
First 100 decoded characters:
Welcome to Natural Language Processing  
It is one of the most exciting research areas as of today  


## 4. GPT Configuration

GPT-2 and GPT-3 are both decoder-only Transformers. The toy GPT-3-like model below is deeper and wider than the toy GPT-2-like model, but both use the same core block.

In [6]:
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0

    @property
    def head_dim(self):
        assert self.n_embd % self.n_head == 0
        return self.n_embd // self.n_head


# Real metadata for discussion only. Do not instantiate these in this notebook.
GPT2_SMALL_REAL = GPTConfig(vocab_size=50_257, block_size=1024, n_layer=12, n_head=12, n_embd=768)
GPT3_175B_REAL = GPTConfig(vocab_size=50_257, block_size=2048, n_layer=96, n_head=96, n_embd=12_288)

# Runnable toy models using this dataset's vocabulary.
GPT2_TOY = GPTConfig(vocab_size=tokenizer.vocab_size, block_size=64, n_layer=2, n_head=4, n_embd=32)
GPT3_TOY = GPTConfig(vocab_size=tokenizer.vocab_size, block_size=64, n_layer=4, n_head=4, n_embd=64)

print("GPT-2 small metadata:", GPT2_SMALL_REAL)
print("GPT-3 175B metadata:", GPT3_175B_REAL)
print("Toy GPT-2-like config:", GPT2_TOY)
print("Toy GPT-3-like config:", GPT3_TOY)

GPT-2 small metadata: GPTConfig(vocab_size=50257, block_size=1024, n_layer=12, n_head=12, n_embd=768, dropout=0.0)
GPT-3 175B metadata: GPTConfig(vocab_size=50257, block_size=2048, n_layer=96, n_head=96, n_embd=12288, dropout=0.0)
Toy GPT-2-like config: GPTConfig(vocab_size=29, block_size=64, n_layer=2, n_head=4, n_embd=32, dropout=0.0)
Toy GPT-3-like config: GPTConfig(vocab_size=29, block_size=64, n_layer=4, n_head=4, n_embd=64, dropout=0.0)


## 5. Causal Multi-Head Self-Attention

Self-attention lets each token gather information from earlier tokens. The causal mask prevents token position `t` from looking at future positions. This is the key rule that allows GPT to generate text left-to-right.

In [7]:
class CausalSelfAttention(nn.Module):
    """Masked multi-head self-attention used in GPT-style models."""

    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.head_dim = config.head_dim

        # This single linear layer computes queries, keys, and values together.
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)

        # Projection after all attention heads are concatenated.
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        # Lower-triangular causal mask. register_buffer stores it with the model
        # but does not treat it as a trainable parameter.
        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer("causal_mask", mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.shape

        # q, k, v each have shape (B, T, C).
        q, k, v = self.qkv(x).split(C, dim=-1)

        # Split the embedding channels across multiple heads:
        # (B, T, C) -> (B, n_head, T, head_dim).
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention.
        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Mask future positions by setting their scores to negative infinity.
        scores = scores.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        # Weighted sum of values, then merge heads back to (B, T, C).
        y = weights @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

## 6. Transformer Decoder Block

A GPT block contains causal self-attention, an MLP, layer normalization, and residual connections.

In [8]:
class FeedForward(nn.Module):
    """Position-wise MLP used after attention."""

    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """One decoder-only Transformer block."""

    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = FeedForward(config)

    def forward(self, x):
        # Pre-layer normalization: normalize first, then apply the sublayer.
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

## 7. Full GPT Language Model

The model maps token IDs to logits. Each logit vector scores every possible next token in the vocabulary.

In [9]:
class GPTLanguageModel(nn.Module):
    """Small GPT-style language model."""

    def __init__(self, config):
        super().__init__()
        self.config = config

        # Token embedding maps token IDs to vectors.
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)

        # Positional embedding tells the model where each token is in the sequence.
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)

        # Stack of Transformer decoder blocks.
        self.blocks = nn.Sequential(*[TransformerBlock(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)

        # Language modeling head maps hidden vectors to vocabulary logits.
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.config.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.config.block_size}")

        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)[None, :, :]
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            # Cross entropy compares predicted next-token logits with true next tokens.
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressively sample new tokens after the input context."""
        self.eval()

        for _ in range(max_new_tokens):
            # Keep only the last block_size tokens as context.
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)

            # Use only the final position to predict the next token.
            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k is not None:
                values, _ = torch.topk(logits, k=top_k)
                cutoff = values[:, [-1]]
                logits = torch.where(logits < cutoff, torch.full_like(logits, float("-inf")), logits)

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)

        return idx

## 8. Instantiate Toy GPT-2-like and GPT-3-like Models

In [10]:
def count_parameters(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


gpt2_model = GPTLanguageModel(GPT2_TOY).to(device)
gpt3_model = GPTLanguageModel(GPT3_TOY).to(device)

print(f"Toy GPT-2-like parameters: {count_parameters(gpt2_model):,}")
print(f"Toy GPT-3-like parameters: {count_parameters(gpt3_model):,}")

Toy GPT-2-like parameters: 29,405
Toy GPT-3-like parameters: 207,901


## 9. Encode a Prompt and Predict the Next Token

The logits from the final prompt position are the model's next-token scores. These random-weight models are not yet trained, so the predictions demonstrate mechanics rather than meaningful language ability.

In [11]:
@torch.no_grad()
def show_next_token_candidates(model, prompt, tokenizer, top_k=5):
    """Print the top next-token candidates for a prompt."""
    model.eval()
    ids = tokenizer.encode(prompt)
    if len(ids) == 0:
        raise ValueError("Prompt has no characters found in the tokenizer vocabulary.")

    idx = torch.tensor(ids, dtype=torch.long, device=device)[None, -model.config.block_size:]
    logits, _ = model(idx)
    probs = F.softmax(logits[0, -1], dim=-1)
    values, indices = torch.topk(probs, k=top_k)

    print("Prompt:")
    print(prompt)
    print("\nEncoded prompt IDs:")
    print(idx[0].tolist())
    print("\nTop next-token candidates:")
    for prob, token_id in zip(values.tolist(), indices.tolist()):
        print(f"id={token_id:>3} token={tokenizer.decode([token_id])!r} probability={prob:.4f}")


prompt = text_data[:80]

print("Toy GPT-2-like next-token prediction")
show_next_token_candidates(gpt2_model, prompt, tokenizer)

print("\nToy GPT-3-like next-token prediction")
show_next_token_candidates(gpt3_model, prompt, tokenizer)

Toy GPT-2-like next-token prediction
Prompt:
Welcome to Natural Language Processing  
It is one of the most exciting research

Encoded prompt IDs:
[8, 18, 1, 4, 8, 20, 14, 25, 8, 14, 12, 1, 6, 22, 21, 10, 12, 23, 23, 16, 20, 14, 1, 1, 0, 3, 24, 1, 16, 23, 1, 21, 20, 12, 1, 21, 13, 1, 24, 15, 12, 1, 19, 21, 23, 24, 1, 12, 27, 10, 16, 24, 16, 20, 14, 1, 22, 12, 23, 12, 8, 22, 10, 15]

Top next-token candidates:
id= 19 token='m' probability=0.1092
id=  9 token='b' probability=0.0732
id= 16 token='i' probability=0.0568
id=  8 token='a' probability=0.0552
id=  0 token='\n' probability=0.0472

Toy GPT-3-like next-token prediction
Prompt:
Welcome to Natural Language Processing  
It is one of the most exciting research

Encoded prompt IDs:
[8, 18, 1, 4, 8, 20, 14, 25, 8, 14, 12, 1, 6, 22, 21, 10, 12, 23, 23, 16, 20, 14, 1, 1, 0, 3, 24, 1, 16, 23, 1, 21, 20, 12, 1, 21, 13, 1, 24, 15, 12, 1, 19, 21, 23, 24, 1, 12, 27, 10, 16, 24, 16, 20, 14, 1, 22, 12, 23, 12, 8, 22, 10, 15]

Top next-token cand

## 10. Generate a Given Length of Text

Change `GENERATE_LENGTH` to control how many new characters are generated.

In [12]:
def generate_text(model, prompt, tokenizer, max_new_tokens=200, temperature=1.0, top_k=10):
    """Encode a prompt, generate new token IDs, and decode back to text."""
    ids = tokenizer.encode(prompt)
    idx = torch.tensor(ids, dtype=torch.long, device=device)[None, :]
    generated = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(generated[0].tolist())


GENERATE_LENGTH = 200

print("Toy GPT-2-like generated text:")
print(generate_text(gpt2_model, prompt, tokenizer, max_new_tokens=GENERATE_LENGTH))

print("\nToy GPT-3-like generated text:")
print(generate_text(gpt3_model, prompt, tokenizer, max_new_tokens=GENERATE_LENGTH))

Toy GPT-2-like generated text:
Welcome to Natural Language Processing  
It is one of the most exciting research
awmbi.Naa cuPclmlmaclyult
buulmcbyabPlytalm tmIabf
ltbtbi tt
Naa cbildufbm
ha
NmbbmbimImlt
bPcbyPgnmmammlNbPIlyt
Plcti
bcmfnaaibPmtbfbcm bi f ubyicicfmbilmf ttuf
urlyPmI
at
hm t
tucNlctmmcff dN NNPIl

Toy GPT-3-like generated text:
Welcome to Natural Language Processing  
It is one of the most exciting researchPWg
eyfg
ygweyfececwhwyIlfWPekecwrecyfegWefg eyWkeyecg
gIPihrWgIuIPfefemly
 eyeWuylgg
 wwauuoanwuoyILhWeeyafgg
NyWIgfyygge
hPee
egyyfauoheggWPgiuIuhWwyliufWuogihgg weyrImlgylecyIcWoiImefWuecwecggWhuuo


## 11. Optional: Mini-Batches for Next-Token Training

For next-token training, the input sequence `x` and target sequence `y` are offset by one token.

In [13]:
def get_batch(data_ids, batch_size=16, block_size=64):
    """Create mini-batches for next-token prediction."""
    if len(data_ids) <= block_size + 1:
        raise ValueError("Dataset is too short for the requested block_size.")

    starts = torch.randint(0, len(data_ids) - block_size - 1, (batch_size,))
    x = torch.stack([data_ids[i : i + block_size] for i in starts])
    y = torch.stack([data_ids[i + 1 : i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)


batch_x, batch_y = get_batch(data_ids, batch_size=2, block_size=GPT2_TOY.block_size)
print("x shape:", tuple(batch_x.shape))
print("y shape:", tuple(batch_y.shape))
print("Example input text:")
print(tokenizer.decode(batch_x[0].cpu().tolist()))
print("Example target text, shifted by one token:")
print(tokenizer.decode(batch_y[0].cpu().tolist()))

x shape: (2, 64)
y shape: (2, 64)
Example input text:
lcome to Natural Language Processing  
It is one of the most exc
Example target text, shifted by one token:
come to Natural Language Processing  
It is one of the most exci


## 12. Optional: Tiny Training Loop

This short loop trains only the toy GPT-2-like model for demonstration. It is not meant to produce a strong language model, but it shows how PyTorch makes training much easier than a pure NumPy implementation.

In [14]:
TRAIN_STEPS = 100
LEARNING_RATE = 3e-4

optimizer = torch.optim.AdamW(gpt2_model.parameters(), lr=LEARNING_RATE)
gpt2_model.train()

for step in range(TRAIN_STEPS):
    x, y = get_batch(data_ids, batch_size=16, block_size=GPT2_TOY.block_size)
    logits, loss = gpt2_model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 25 == 0 or step == TRAIN_STEPS - 1:
        print(f"step {step:>3} | loss {loss.item():.4f}")

print("\nToy GPT-2-like generated text after tiny training loop:")
print(generate_text(gpt2_model, prompt, tokenizer, max_new_tokens=GENERATE_LENGTH))


step   0 | loss 3.5213
step  25 | loss 3.2238
step  50 | loss 2.9228
step  75 | loss 2.7883
step  99 | loss 2.6582

Toy GPT-2-like generated text after tiny training loop:
Welcome to Natural Language Processing  
It is one of the most exciting researchm maodw  oe mos c soso 
n s iis e atli  ooossa   mts  tt
nes eiod od
h 
ast as o ea s is m tt tes ti  i c  ca t

ai otha eeiao on oo   s
n coes s ts amiie  theloels otosni aro  smi 
We  t e o mes ee a


In [15]:
TRAIN_STEPS = 100
LEARNING_RATE = 3e-4

optimizer = torch.optim.AdamW(gpt3_model.parameters(), lr=LEARNING_RATE)
gpt3_model.train()

for step in range(TRAIN_STEPS):
    x, y = get_batch(data_ids, batch_size=16, block_size=GPT3_TOY.block_size)
    logits, loss = gpt3_model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 25 == 0 or step == TRAIN_STEPS - 1:
        print(f"step {step:>3} | loss {loss.item():.4f}")

print("\nToy GPT-3-like generated text after tiny training loop:")
print(generate_text(gpt3_model, prompt, tokenizer, max_new_tokens=GENERATE_LENGTH))

step   0 | loss 3.5872
step  25 | loss 2.6990
step  50 | loss 2.3592
step  75 | loss 2.0089
step  99 | loss 1.8665

Toy GPT-3-like generated text after tiny training loop:
Welcome to Natural Language Processing  
It is one of the most exciting researchowofow Parcang as 
Itis af   earare onge tof 
Will eaaf e arcarche hhheiodash arce oday owis o eaf e 
foy the  aaf ton 
f oftitesth w Py of hor oure t ines e e 
Woy  eas it PWeare eseasoss ayod Paresc


## 13. Save the Trained Toy GPT-3 Model

After training, save a checkpoint that contains the model weights, model configuration, and tokenizer vocabulary. The configuration and vocabulary are needed because the saved weights only make sense for the exact same architecture and token-to-ID mapping.

In [ ]:
CHECKPOINT_PATH = Path("toy_gpt3_char_model.pt")

gpt3_checkpoint = {
    # Learned PyTorch parameters for the trained Toy GPT-3-like model.
    "model_state_dict": gpt3_model.state_dict(),

    # Store the architecture settings so the model can be rebuilt later.
    "config": GPT3_TOY.__dict__,

    # Store the character vocabulary so token IDs mean the same thing at inference.
    "chars": tokenizer.chars,
}

torch.save(gpt3_checkpoint, CHECKPOINT_PATH)
print(f"Saved Toy GPT-3-like checkpoint to: {CHECKPOINT_PATH.resolve()}")

## 14. Load the Saved Model for Inference

This cell reconstructs the tokenizer and model from the checkpoint, loads the saved weights, switches the model to evaluation mode, and then generates text from the same prompt.

In [ ]:
class LoadedCharTokenizer:
    """Tokenizer reconstructed from the saved checkpoint vocabulary."""

    def __init__(self, chars):
        self.chars = list(chars)
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

    @property
    def vocab_size(self):
        return len(self.chars)

    def encode(self, text):
        return [self.stoi[ch] for ch in text if ch in self.stoi]

    def decode(self, ids):
        return "".join(self.itos[int(i)] for i in ids)


loaded_checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
loaded_config = GPTConfig(**loaded_checkpoint["config"])
loaded_tokenizer = LoadedCharTokenizer(loaded_checkpoint["chars"])

loaded_gpt3_model = GPTLanguageModel(loaded_config).to(device)
loaded_gpt3_model.load_state_dict(loaded_checkpoint["model_state_dict"])
loaded_gpt3_model.eval()

print("Loaded model config:", loaded_config)
print(f"Loaded model parameters: {count_parameters(loaded_gpt3_model):,}")
print(f"Loaded tokenizer vocabulary size: {loaded_tokenizer.vocab_size}")

## 15. Inference with the Loaded Toy GPT-3 Model

The loaded model can now be used without retraining. This is the typical workflow for deployment or later inference: define the architecture, load the checkpoint, call `eval()`, and generate under `torch.no_grad()`.

In [ ]:
print("Loaded Toy GPT-3-like next-token prediction:")
show_next_token_candidates(loaded_gpt3_model, prompt, loaded_tokenizer)

print("\nLoaded Toy GPT-3-like generated text:")
print(generate_text(
    loaded_gpt3_model,
    prompt,
    loaded_tokenizer,
    max_new_tokens=GENERATE_LENGTH,
    temperature=1.0,
    top_k=10,
))

## 16. Journal Club Discussion Prompts

1. What parts of this PyTorch implementation correspond directly to the NumPy version?
2. Why does causal masking prevent information leakage during generation?
3. Why do GPT-2 and GPT-3 differ mainly by scale rather than by basic architecture?
4. What changes would be needed to move from character tokens to GPT-style subword tokens?
5. What are the limitations of using a text-only model for biomedical imaging or multimodal research?